<a href="https://colab.research.google.com/github/Deangr-econ/torch-volatility-engine/blob/main/HAR_Markov_Switching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd
from google.colab import userdata
import torch
import time
import numba
from scipy.optimize import minimize
# from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

# 1. Load data
data_futures = pd.read_csv(userdata.get('Future_data'))

# 2. Filter symbol & set date index directly
df = (
    data_futures.query("symbol == 'NG'")
    .set_index('date')[['rv5', 'rsp5', 'rsn5']]
)

# 3. Feature engineering (HAR daily, weekly, monthly features)
data = df[['rsp5', 'rsn5', 'rv5']].copy()

for feature in ['rsp5', 'rsn5', 'rv5']:
    data[f'{feature}_w'] = data[feature].rolling(5).mean()   # Added ()
    data[f'{feature}_m'] = data[feature].rolling(22).mean()  # Added ()

# Lag all features by 1 day to prevent look-ahead bias
data = data.shift(1)

# Add target column back and drop initial rolling NaNs
data['rv5_p'] = df['rv5']
data = data.dropna()

# 4. Define target & predictors dynamically
all_features = [col for col in data.columns if col != 'rv5_p']
y = data['rv5_p']
X = data[all_features].assign(const=1.0)[['const'] + all_features]

# 5. Solve for OLS parameters & predict
beta = np.linalg.solve(X.T @ X, X.T @ y)
pred = X.values @ beta

# 6. Evaluation metrics
e = y - pred
rmsfe = np.sqrt((e**2).mean())

ss_res = np.sum(e**2)
ss_tot = np.sum((y - y.mean())**2)
r_squared = 1 - (ss_res / ss_tot)

print(f"RMSFE: {rmsfe:.6f}")
print(f"R²:    {r_squared:.4f}")

RMSFE: 0.001653
R²:    0.1436


In [8]:
data

,rsp5,rsn5,rv5,rsp5_w,rsp5_m,rsn5_w,rsn5_m,rv5_w,rv5_m,rv5_p
date,,,,,,,,,,
2009-10-28,0.000664,0.000437,0.001101,0.000713,0.003782,0.000838,0.000873,0.001551,0.004656,0.030706
2009-10-29,0.029650,0.001056,0.030706,0.006436,0.002141,0.000806,0.000857,0.007243,0.002998,0.001235
2009-10-30,0.000625,0.000610,0.001235,0.006380,0.002108,0.000740,0.000842,0.007120,0.002950,0.001344
2009-11-02,0.000568,0.000776,0.001344,0.006410,0.002105,0.000763,0.000847,0.007173,0.002952,0.001318
2009-11-03,0.000399,0.000919,0.001318,0.006381,0.002095,0.000760,0.000826,0.007141,0.002921,0.000886
...,...,...,...,...,...,...,...,...,...,...
2026-09-24,0.002027,0.000380,0.002407,0.000617,0.000345,0.000246,0.000261,0.000863,0.000606,0.001948
2026-09-25,0.001128,0.000820,0.001948,0.000800,0.000384,0.000365,0.000284,0.001165,0.000668,0.000915
2026-09-28,0.000440,0.000474,0.000915,0.000833,0.000393,0.000422,0.000298,0.001256,0.000691,0.000831


In [11]:
def build_tensor(df, target_col='rv5', train_pct=0.80):
    """
    Packs expanding window feature matrices and target vectors into 3D tensors.

    Returns:
        X_3d: Tensor of shape (test_size, max_window_len, K)
        y_3d: Tensor of shape (test_size, max_window_len)
        X_test: Out-of-sample feature vectors of shape (test_size, 1, K)
        y_test: Actual out-of-sample target values of shape (test_size,)
    """
    # 1. Separate features and target
    feature_cols = [col for col in df.columns if col != target_col]

    # Extract values and create constant term for X
    y_tensor = torch.tensor(df[target_col].values, dtype=torch.float64)

    X_df = df[feature_cols].assign(const=1.0)[['const'] + feature_cols]
    X_tensor = torch.tensor(X_df.values, dtype=torch.float64)

    # 2. Calculate dimensions
    N, K = X_tensor.shape
    train_split = int(train_pct * N)
    test_size = N - train_split
    max_window_len = N - 1  # Maximum row length across all expanding windows

    # 3. Initialize 3D containers (padded with zeros)
    X_3d = torch.zeros((test_size, max_window_len, K), dtype=torch.float64)
    y_3d = torch.zeros((test_size, max_window_len), dtype=torch.float64)

    # 4. Fill expanding window slices (LEGO blocks)
    for i in range(test_size):
        cur_len = train_split + i
        X_3d[i, :cur_len, :] = X_tensor[:cur_len, :]
        y_3d[i, :cur_len] = y_tensor[:cur_len]

    X_test = X_tensor[train_split:N, :].unsqueeze(1)
    y_test = y_tensor[train_split:N]

    return X_3d, y_3d, X_test, y_test


def solve_expanding_ols(X_3d, y_3d, X_test):
    """
    Solves M expanding-window OLS regressions simultaneously and generates
    1-step-ahead out-of-sample forecasts using PyTorch batched tensor ops.

    Parameters:
    -----------
    X_3d   : Tensor of shape (M, max_window_len, K)
    y_3d   : Tensor of shape (M, max_window_len)
    X_test : Tensor of shape (M, 1, K)

    Returns:
    --------
    y_pred : Tensor of shape (M,) containing 1-step-ahead forecasts
    betas  : Tensor of shape (M, K) containing estimated coefficients for all windows
    """
    # 1. Reshape y_3d to (M, max_window_len, 1) for matrix multiplication
    y_3d_col = y_3d.unsqueeze(-1)

    # 2. Compute X^T * X for all M windows simultaneously -> Shape: (M, K, K)
    # transpose(1, 2) flips (max_window_len, K) to (K, max_window_len)
    XtX = torch.bmm(X_3d.transpose(1, 2), X_3d)

    # 3. Compute X^T * y for all M windows simultaneously -> Shape: (M, K, 1)
    Xty = torch.bmm(X_3d.transpose(1, 2), y_3d_col)

    # 4. Solve XtX * beta = Xty for beta across all M models -> Shape: (M, K, 1)
    betas_col = torch.linalg.solve(XtX, Xty)

    # 5. Generate 1-step-ahead predictions: X_test * beta -> Shape: (M, 1, 1)
    y_pred_3d = torch.bmm(X_test, betas_col)

    # 6. Flatten dimensions back to clean 1D and 2D arrays
    y_pred = y_pred_3d.squeeze()       # Shape: (M,)
    betas = betas_col.squeeze(-1)       # Shape: (M, K)

    return y_pred, betas

def rmsfe(y_true, y_pred):
  return torch.sqrt(torch.mean((y_true - y_pred) ** 2)).item()

def FSP(df, target_col='rv5', train_pct=0.80):
    X_3d, y_3d, X_test, y_test = build_tensor(df, target_col, train_pct)
    y_pred, betas = solve_expanding_ols(X_3d, y_3d, X_test)
    rmsfe_val = rmsfe(y_test, y_pred)

    return rmsfe_val, y_pred, betas, y_test

def get_forecast_metrics(y_true, y_pred, is_log=False):
    """
    Computes core out-of-sample volatility forecasting performance metrics.

    Parameters:
    -----------
    y_true : array-like
        Actual volatility series.
    y_pred : array-like
        Forecasted volatility series.
    is_log : bool, optional
        If True, exponentiates both arrays before calculating metrics to evaluate on original scale.

    Returns:
    --------
    dict
        A dictionary containing RMSE, MAE, and QLIKE values.
    """
    # Convert to numpy arrays if necessary
    y_t = np.array(y_true)
    y_p = np.array(y_pred)

    if is_log:
        y_t = np.exp(y_t)
        y_p = np.exp(y_p)

    rmse = np.sqrt(np.mean((y_t - y_p) ** 2))
    mae = np.mean(np.abs(y_t - y_p))

    # QLIKE loss: standard asymmetric loss function for volatility forecasts
    # Handles dividing by zero with safety checks
    with np.errstate(divide='ignore', invalid='ignore'):
        qlike = np.mean(y_t / y_p - np.log(y_t / y_p) - 1.0) if np.all(y_p > 0) else np.nan

    return {
        'RMSE': float(rmse),
        'MAE': float(mae),
        'QLIKE': float(qlike)
    }

# Example of usage:
# metrics = get_forecast_metrics(y_test_base_np, pred_base_ols_np)
# print(metrics)

In [18]:
# =============================================================================
# 1. GENERAL M-REGIME COMPILED HAMILTON LOG-LIKELIHOOD KERNEL
# =============================================================================
@numba.njit(fastmath=True)
def fast_hamilton_ll_general(mu_mat, y, sigmas, P):
    """
    General M-regime Hamilton Filter Kernel.

    mu_mat : (T, M) - conditional means per regime
    y      : (T,)   - observed target series
    sigmas : (M,)   - error standard deviations per regime
    P      : (M, M) - transition matrix where P[i, j] = Prob(S_t = j | S_{t-1} = i)
    """
    T, M = mu_mat.shape
    inv_sigmas = 1.0 / (sigmas * 2.506628274631)

    # -------------------------------------------------------------
    # Stationary / Ergodic Steady-State Prior: (P.T - I) * pi = 0
    # -------------------------------------------------------------
    A = np.copy(P.T)
    for m in range(M):
        A[m, m] -= 1.0
    for m in range(M):
        A[M - 1, m] = 1.0  # Replace last equation with sum(pi) = 1

    b = np.zeros(M)
    b[M - 1] = 1.0

    # Solve linear system; fallback to uniform prior if singular
    try:
        xi = np.linalg.solve(A, b)
        for m in range(M):
            if xi[m] < 0.0 or np.isnan(xi[m]):
                xi = np.ones(M) / M
                break
    except:
        xi = np.ones(M) / M

    ll = 0.0
    pr = np.zeros(M)
    eta = np.zeros(M)

    # -------------------------------------------------------------
    # Main Filter Time Loop
    # -------------------------------------------------------------
    for t in range(T):
        # 1. State projection: pr = P.T @ xi
        for j in range(M):
            acc = 0.0
            for i in range(M):
                acc += P[i, j] * xi[i]
            pr[j] = acc

        # 2. Gaussian conditional densities
        yt = y[t]
        for m in range(M):
            e = (yt - mu_mat[t, m]) / sigmas[m]
            eta[m] = inv_sigmas[m] * np.exp(-0.5 * e * e)

        # 3. Mixture density
        f = 0.0
        for m in range(M):
            f += pr[m] * eta[m]

        if f < 1e-300:
            f = 1e-300
        ll += np.log(f)

        # 4. Bayesian Posterior Update
        inv_f = 1.0 / f
        for m in range(M):
            xi[m] = (pr[m] * eta[m]) * inv_f

    return ll, xi


# =============================================================================
# 2. MATRIX TRANSFORMATIONS & UNCONSTRAINED REPARAMETERIZATION
# =============================================================================
def logits_to_P(logits_matrix):
    """
    Numerically stable row-wise Softmax: converts unconstrained (M, M) logits
    into a valid row-stochastic transition matrix P where each row sums to 1.
    """
    shift = logits_matrix - np.max(logits_matrix, axis=1, keepdims=True)
    exp_logits = np.exp(shift)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def P_to_logits(P):
    """
    Converts a valid transition matrix P to logit space.
    """
    return np.log(np.clip(P, 1e-6, 1.0 - 1e-6))


# =============================================================================
# 3. GENERAL OBJECTIVE WRAPPER
# =============================================================================
def objective_hamilton_general(theta, X, y, K, M):
    """
    Layout of theta:
      - theta[0 : M*K]               -> Betas (K x M)
      - theta[M*K : M*K + M]         -> Sigmas (M,)
      - theta[M*K + M : M*K + M + M^2] -> Transition matrix logits (M x M)
    """
    n_betas = M * K
    betas = theta[:n_betas].reshape(K, M)
    sigmas = theta[n_betas : n_betas + M]
    logits = theta[n_betas + M : n_betas + M + (M * M)].reshape(M, M)

    # Boundary penalty
    if np.any(sigmas <= 1e-5):
        return 1e12

    P = logits_to_P(logits)

    # Pre-calculate all M regime paths simultaneously: (T, M)
    mu_mat = X @ betas

    ll, _ = fast_hamilton_ll_general(mu_mat, y, sigmas, P)

    if np.isnan(ll) or np.isinf(ll):
        return 1e12

    return -ll


# =============================================================================
# 4. INITIALIZATION & SINGLE-WINDOW ESTIMATION
# =============================================================================
def get_initial_theta_general(X_init, y_init, K, M):
    """
    Derives warm-start parameters for arbitrary M regimes from baseline OLS.
    """
    beta_ols = np.linalg.lstsq(X_init, y_init, rcond=None)[0]
    resids = y_init - X_init @ beta_ols
    s_ols = float(np.std(resids))

    # Multipliers spanning lower to higher volatility regimes
    sigma_scales = np.linspace(0.70, 1.60, M)
    beta_scales = np.linspace(0.95, 1.05, M)

    betas_init = np.outer(beta_ols, beta_scales)  # Shape (K, M)
    sigmas_init = s_ols * sigma_scales            # Shape (M,)

    # Sticky diagonal transition matrix prior (e.g., 0.90 on diagonal)
    P_init = np.full((M, M), (1.0 - 0.90) / (M - 1))
    np.fill_diagonal(P_init, 0.90)
    logits_init = P_to_logits(P_init)

    return np.concatenate([
        betas_init.ravel(),
        sigmas_init,
        logits_init.ravel()
    ])


def fit_window_general(theta_start, X_train, y_train, K, M, bounds, maxiter=50):
    """
    Fits one window using SciPy L-BFGS-B with arbitrary M regimes.
    """
    opt_res = minimize(
        fun=objective_hamilton_general,
        x0=theta_start,
        args=(X_train, y_train, K, M),
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': maxiter, 'ftol': 1e-6, 'gtol': 1e-4}
    )
    if opt_res.success or opt_res.fun < 1e10:
        return opt_res.x, opt_res.fun
    return theta_start, opt_res.fun


# =============================================================================
# 5. 1-STEP FORECASTING PROJECTION
# =============================================================================
def predict_one_step_general(theta, X_train, y_train, x_next, K, M):
    """
    Extracts terminal filtered state, projects forward via P.T, and computes
    1-step ahead expected conditional variance.
    """
    n_betas = M * K
    betas = theta[:n_betas].reshape(K, M)
    sigmas = theta[n_betas : n_betas + M]
    logits = theta[n_betas + M : n_betas + M + (M * M)].reshape(M, M)
    P = logits_to_P(logits)

    # 1. In-sample fit & terminal filtered state extraction
    mu_mat = X_train @ betas
    _, xi_T = fast_hamilton_ll_general(mu_mat, y_train, sigmas, P)

    # 2. Project state forward 1 step: xi_{t+1|t} = P.T @ xi_T
    xi_next = P.T @ xi_T

    # 3. Compute conditional expectations for tomorrow: (M,)
    mu_next = x_next @ betas

    # 4. Final mixture expectation
    y_hat = np.dot(xi_next, mu_next)

    return y_hat, xi_next


def evaluate_forecasts(y_true, y_pred, is_log=False):
    """
    Computes standard volatility loss metrics: RMSE, MAE, and QLIKE.
    """
    y_t = np.exp(y_true) if is_log else y_true
    y_p = np.exp(y_pred) if is_log else y_pred

    rmse = np.sqrt(np.mean((y_t - y_p) ** 2))
    mae = np.mean(np.abs(y_t - y_p))
    qlike = np.mean(y_t / y_p - np.log(y_t / y_p) - 1.0) if np.all(y_p > 0) else np.nan

    return {'RMSE': rmse, 'MAE': mae, 'QLIKE': qlike}


# =============================================================================
# 6. MASTER ENGINE: run_ms_har_forecasting
# =============================================================================
def run_ms_har_forecasting(
    data,
    target_col,
    exog_cols=None,
    train_split=0.80,
    n_regimes=2,
    is_log=True,
    verbose=True
):
    """
    Executes an expanding window out-of-sample forecasting pipeline using an MS-HAR model
    generalized to an arbitrary number of regimes M.

    Parameters:
    -----------
    data : pd.DataFrame
        Full dataset containing features and target.
    target_col : str
        Name of the dependent variable column.
    exog_cols : list of str, optional
        List of exogenous feature column names. If None, uses all columns except target_col.
    train_split : float, optional
        Fraction of data used for initial in-sample calibration (default: 0.80).
    n_regimes : int, optional
        Number of regimes M (e.g., 2, 3, 4).
    is_log : bool, optional
        Whether the target variable is modeled in logarithms.
    verbose : bool, optional
        Whether to print rolling execution progress.

    Returns:
    --------
    dict containing:
        - 'forecasts': np.ndarray of out-of-sample predictions
        - 'regime_probs': np.ndarray of shape (N_oos, M) with predicted regime probabilities
        - 'param_history': np.ndarray of shape (N_oos, n_params) with estimated parameters
        - 'metrics': dict with RMSE, MAE, QLIKE
        - 'y_true': np.ndarray of actual out-of-sample targets
        - 'total_time': total execution time in seconds
    """
    M = int(n_regimes)
    if M < 2:
        raise ValueError("n_regimes must be >= 2.")

    # 1. Feature matrix & Target vector preparation
    if exog_cols is None:
        exog_cols = [c for c in data.columns if c != target_col]

    X_full = np.ascontiguousarray(data[exog_cols].values, dtype=np.float64)
    y_full = np.ascontiguousarray(data[target_col].values, dtype=np.float64)

    N_total, K = X_full.shape
    T_start = int(N_total * train_split)
    n_forecasts = N_total - T_start
    n_params = (M * K) + M + (M * M)

    if verbose:
        print(f"Model Configuration: {M} Regimes | {K} Exogenous Variables | {n_params} Parameters")
        print(f"Total Observations: {N_total} | Initial Calibration: {T_start} | Forecast Horizon: {n_forecasts}")

    # 2. Box bounds for L-BFGS-B
    # Betas: unconstrained
    # Sigmas: strictly positive [1e-4, 50.0]
    # Logits: unconstrained [-15.0, 15.0] (prevents numerical overflow in Softmax)
    bounds = (
        [(None, None)] * (M * K) +
        [(1e-4, 50.0)] * M +
        [(-15.0, 15.0)] * (M * M)
    )

    # 3. Initial Window Calibration (Cold Start)
    if verbose:
        print("Calibrating initial window (Cold Start)...")
    theta_init = get_initial_theta_general(X_full[:T_start], y_full[:T_start], K, M)
    current_theta, init_ll = fit_window_general(
        theta_init, X_full[:T_start], y_full[:T_start], K, M, bounds, maxiter=150
    )

    # 4. Pre-allocate Output Arrays
    y_hat_oos = np.zeros(n_forecasts, dtype=np.float64)
    regime_prob_oos = np.zeros((n_forecasts, M), dtype=np.float64)
    param_history = np.zeros((n_forecasts, n_params), dtype=np.float64)

    # 5. Expanding Window Rolling Loop
    start_time = time.time()
    for i in range(n_forecasts):
        t_end = T_start + i

        X_train = X_full[:t_end]
        y_train = y_full[:t_end]
        x_next = X_full[t_end]

        # Warm-started optimization
        current_theta, fun_val = fit_window_general(
            current_theta, X_train, y_train, K, M, bounds, maxiter=50
        )
        param_history[i] = current_theta

        # 1-step forecast & state projection
        y_hat, probs = predict_one_step_general(current_theta, X_train, y_train, x_next, K, M)
        y_hat_oos[i] = y_hat
        regime_prob_oos[i] = probs

        if verbose and ((i + 1) % 100 == 0 or i == 0 or i == n_forecasts - 1):
            elapsed = time.time() - start_time
            speed = (i + 1) / elapsed
            print(f"Step {i+1:3d}/{n_forecasts} | Elapsed: {elapsed:5.1f}s | Speed: {speed:4.1f} win/s | -LL: {fun_val:7.2f}")

    total_time = time.time() - start_time
    y_true_oos = y_full[T_start:]

    # 6. Evaluation
    metrics = evaluate_forecasts(y_true_oos, y_hat_oos, is_log=is_log)

    if verbose:
        print(f"\nExecution Complete in {total_time:.2f}s ({total_time / n_forecasts * 1000:.1f} ms/window).")
        print(f"RMSE:  {metrics['RMSE']:.6f} | MAE: {metrics['MAE']:.6f} | QLIKE: {metrics['QLIKE']:.6f}")

    return {
        # 'forecasts': y_hat_oos,
        # 'regime_probs': regime_prob_oos,
        # 'param_history': param_history,
        'metrics': metrics,
        # 'y_true': y_true_oos,
        'total_time': total_time
    }

In [19]:
data_model = data.copy()

# Configure which regimes you want to dynamically test/compare
regimes_to_test = [*range(2,6)]

# Define the three different predictor groups
predictor_groups = {
    'rv5 Only': ['rv5'],
    'rv5 HAR': ['rv5', 'rv5_w', 'rv5_m'],
    'RS HAR': ['rsp5', 'rsn5', 'rsp5_w', 'rsn5_w', 'rsp5_m', 'rsn5_m']
}

results_dict_rv = {}

for group_name, exog_cols in predictor_groups.items():
    # -----------------------------------------------------
    # 1. Base OLS Estimation (No regimes)
    # -----------------------------------------------------
    print(f"Estimating OLS Model for {group_name}...")
    df_subset = data_model[['rv5_p'] + exog_cols].copy()
    _, pred_tensor, _, y_test_tensor = FSP(df_subset, target_col='rv5_p', train_pct=0.80)

    metrics_ols = get_forecast_metrics(y_test_tensor.numpy(), pred_tensor.numpy(), is_log=False)
    results_dict_rv[f'OLS: {group_name}'] = metrics_ols

    # -----------------------------------------------------
    # 2. Dynamic Regime Switching Models
    # -----------------------------------------------------
    for r in regimes_to_test:
        print(f"Estimating {r}-Regime Model for {group_name}...")
        res_rs = run_ms_har_forecasting(
            data=data_model,
            target_col='rv5_p',
            exog_cols=exog_cols,
            train_split=0.80,
            n_regimes=r,
            is_log=False,
            verbose=False
        )
        results_dict_rv[f'RS ({r} Regimes): {group_name}'] = res_rs['metrics']

# =====================================================
# Build Comparison Matrix
# =====================================================
comparison_matrix_rv = pd.DataFrame(results_dict_rv).T
print("\n=================== COMPARISON MATRIX (LEVEL RV) ===================")
display(comparison_matrix_rv.style.highlight_min(axis=0, color='lightgreen'))

Estimating OLS Model for rv5 Only...
Estimating 2-Regime Model for rv5 Only...
Estimating 3-Regime Model for rv5 Only...
Estimating 4-Regime Model for rv5 Only...
Estimating 5-Regime Model for rv5 Only...
Estimating OLS Model for rv5 HAR...
Estimating 2-Regime Model for rv5 HAR...
Estimating 3-Regime Model for rv5 HAR...
Estimating 4-Regime Model for rv5 HAR...
Estimating 5-Regime Model for rv5 HAR...
Estimating OLS Model for RS HAR...
Estimating 2-Regime Model for RS HAR...
Estimating 3-Regime Model for RS HAR...
Estimating 4-Regime Model for RS HAR...
Estimating 5-Regime Model for RS HAR...

=================== COMPARISON MATRIX (LEVEL RV) ===================


,RMSE,MAE,QLIKE
OLS: rv5 Only,0.002154,0.000648,0.277352
RS (2 Regimes): rv5 Only,0.002204,0.000680,0.380744
RS (3 Regimes): rv5 Only,0.002159,0.000651,0.334723
RS (4 Regimes): rv5 Only,0.002137,0.000639,0.319033
RS (5 Regimes): rv5 Only,0.002128,0.000636,0.314563
OLS: rv5 HAR,0.002107,0.000670,0.235017
RS (2 Regimes): rv5 HAR,0.002109,0.000647,0.236042
RS (3 Regimes): rv5 HAR,0.002098,0.000625,0.233934
RS (4 Regimes): rv5 HAR,0.002085,0.000606,0.225583
RS (5 Regimes): rv5 HAR,0.002078,0.000588,0.223744


In [20]:
import numpy as np
import pandas as pd
import torch

data_model = np.log(data).copy()

# Configure which regimes you want to dynamically test/compare
regimes_to_test = [*range(2,6)]

# Define the three different predictor groups
predictor_groups = {
    'rv5 Only': ['rv5'],
    'rv5 HAR': ['rv5', 'rv5_w', 'rv5_m'],
    'RS HAR': ['rsp5', 'rsn5', 'rsp5_w', 'rsn5_w', 'rsp5_m', 'rsn5_m']
}

results_dict_log_rv = {}

for group_name, exog_cols in predictor_groups.items():
    # -----------------------------------------------------
    # 1. Base OLS Estimation (No regimes)
    # -----------------------------------------------------
    print(f"Estimating OLS Model for {group_name}...")
    df_subset = data_model[['rv5_p'] + exog_cols].copy()
    _, pred_tensor, _, y_test_tensor = FSP(df_subset, target_col='rv5_p', train_pct=0.80)

    # FIXED: Since df_subset is logged, we must pass is_log=True to get level-scale metrics!
    metrics_ols = get_forecast_metrics(y_test_tensor.numpy(), pred_tensor.numpy(), is_log=True)
    results_dict_log_rv[f'OLS: {group_name}'] = metrics_ols

    # -----------------------------------------------------
    # 2. Dynamic Regime Switching Models
    # -----------------------------------------------------
    for r in regimes_to_test:
        print(f"Estimating {r}-Regime Model for {group_name}...")
        res_rs = run_ms_har_forecasting(
            data=data_model,
            target_col='rv5_p',
            exog_cols=exog_cols,
            train_split=0.80,
            n_regimes=r,
            is_log=True,
            verbose=False
        )
        results_dict_log_rv[f'RS ({r} Regimes): {group_name}'] = res_rs['metrics']

# =====================================================
# Build Comparison Matrix
# =====================================================
comparison_matrix_log_rv = pd.DataFrame(results_dict_log_rv).T
print("\n=================== COMPARISON MATRIX (LOG RV) ===================")
display(comparison_matrix_log_rv.style.highlight_min(axis=0, color='lightgreen'))

Estimating OLS Model for rv5 Only...
Estimating 2-Regime Model for rv5 Only...
Estimating 3-Regime Model for rv5 Only...
Estimating 4-Regime Model for rv5 Only...
Estimating 5-Regime Model for rv5 Only...
Estimating OLS Model for rv5 HAR...
Estimating 2-Regime Model for rv5 HAR...
Estimating 3-Regime Model for rv5 HAR...
Estimating 4-Regime Model for rv5 HAR...
Estimating 5-Regime Model for rv5 HAR...
Estimating OLS Model for RS HAR...
Estimating 2-Regime Model for RS HAR...
Estimating 3-Regime Model for RS HAR...
Estimating 4-Regime Model for RS HAR...
Estimating 5-Regime Model for RS HAR...

=================== COMPARISON MATRIX (LOG RV) ===================


,RMSE,MAE,QLIKE
OLS: rv5 Only,0.002192,0.000642,0.281249
RS (2 Regimes): rv5 Only,0.002849,0.000797,0.261951
RS (3 Regimes): rv5 Only,0.002870,0.000800,0.262065
RS (4 Regimes): rv5 Only,0.002875,0.000801,0.262108
RS (5 Regimes): rv5 Only,0.002873,0.000801,0.262089
OLS: rv5 HAR,0.002098,0.000600,0.238674
RS (2 Regimes): rv5 HAR,0.002080,0.000597,0.221870
RS (3 Regimes): rv5 HAR,0.002077,0.000578,0.222288
RS (4 Regimes): rv5 HAR,0.002070,0.000568,0.213170
RS (5 Regimes): rv5 HAR,0.002068,0.000567,0.212682


In [21]:
import pandas as pd
import numpy as np

def create_global_comparison():
    """
    Combines, labels, and formats performance matrices from Level-scale
    and Log-scale evaluations dynamically from the explicit user-defined
    comparison matrices.
    """
    # 1. Retrieve Level results
    df_level = pd.DataFrame()
    if 'comparison_matrix_rv' in globals() and comparison_matrix_rv is not None:
        df_level = comparison_matrix_rv.copy()
        df_level['Scale'] = 'Pure RV (Level)'
        df_level = df_level.reset_index().rename(columns={'index': 'Model'})

    # 2. Retrieve Log results
    df_log = pd.DataFrame()
    if 'comparison_matrix_log_rv' in globals() and comparison_matrix_log_rv is not None:
        df_log = comparison_matrix_log_rv.copy()
        df_log['Scale'] = 'Log RV'
        df_log = df_log.reset_index().rename(columns={'index': 'Model'})

    # 3. Combine both matrices
    if df_level.empty and df_log.empty:
        return pd.DataFrame()

    combined = pd.concat([df_level, df_log], axis=0, ignore_index=True)

    # Reorder and filter columns dynamically
    cols = ['Scale', 'Model', 'QLIKE', 'RMSE', 'MAE']
    available_cols = [c for c in cols if c in combined.columns]
    combined = combined[available_cols]

    # Sort globally by QLIKE
    if 'QLIKE' in combined.columns:
        combined = combined.sort_values(by='QLIKE', ascending=True).reset_index(drop=True)

    return combined

# Build the unified DataFrame dynamically
combined_df_sorted = create_global_comparison()

# Display the sorted unified matrix
print("\n=================== GLOBAL MODEL COMPARISON (SORTED BY QLIKE) ===================")
if not combined_df_sorted.empty:
    subset_cols = [c for c in ['QLIKE', 'RMSE', 'MAE'] if c in combined_df_sorted.columns]
    display(combined_df_sorted.style.highlight_min(subset=subset_cols, axis=0, color='lightgreen'))
else:
    print("No data available to compare. Please execute comparison_matrix_rv and comparison_matrix_log_rv cells first.")


=================== GLOBAL MODEL COMPARISON (SORTED BY QLIKE) ===================


,Scale,Model,QLIKE,RMSE,MAE
0,Log RV,RS (5 Regimes): rv5 HAR,0.212682,0.002068,0.000567
1,Log RV,RS (4 Regimes): rv5 HAR,0.213170,0.002070,0.000568
2,Log RV,RS (3 Regimes): RS HAR,0.217048,0.002071,0.000559
3,Log RV,RS (4 Regimes): RS HAR,0.217439,0.002072,0.000561
4,Log RV,RS (5 Regimes): RS HAR,0.218522,0.002075,0.000558
5,Pure RV (Level),OLS: RS HAR,0.220045,0.002125,0.000660
6,Log RV,RS (2 Regimes): rv5 HAR,0.221870,0.002080,0.000597
7,Log RV,RS (3 Regimes): rv5 HAR,0.222288,0.002077,0.000578
8,Pure RV (Level),RS (5 Regimes): rv5 HAR,0.223744,0.002078,0.000588
9,Log RV,RS (2 Regimes): RS HAR,0.224014,0.002083,0.000572
